# **Stroke Prediction on Imbalanced Data: End-to-End Pipeline**

## **Problem Definition**

**Aims:**

- Build a binary classifier to predict the likelihood of stroke based on demographic, medical, and lifestyle features.
- Support early screening by identifying high-risk individuals using health data.
- Ensure the model is both accurate, robust to imbalance, and clinically interpretable.

---

## **Exploration Data Analysis and Medical Challenges**

1. **Severe Class Imbalance**
   - Only 4.9% of patients had a stroke, biasing predictions toward the majority class.
   - Performance measured by **Recall**, **F1-score**, and **AUC**, rather than Accuracy.

2. **Missing and Ambiguous Values**
   - `bmi` has **3.9% missing values**
   - `smoking_status` has **30% labeled "Unknown"**

3. **Nonlinear Feature Relationships**
   - Stroke risk rises sharply after age 60, glucose >150, and higher BMI.
   - Motivates the use of nonlinear models (e.g., Random Forest, XGBoost).

4. **Categorical Variable Handling**
   - Categorical features like `gender`, `residence_type`, and `work_type` require careful encoding.
   - Poor encoding could lead to overfitting or reduced performance.

5. **Need for Interpretability**
   - Clinical interpretability is vital; methods such as SHAP and decision-tree inspection increase trust.


---

#### **EDA Insights**
Stroke cases are linked with **older age**, **high glucose**, and **slightly higher BMI**. These insights, along with imbalance and clinical needs, suggest using models that are **nonlinear**, **balanced**, and **explainable**.


---

## **Data Processing Strategy**

Data preprocessing strategy was guided by EDA and medical articles cited in the coursework. 

1. **Row Filtering**
   - Excluded `gender == "Other"` and any row with missing values.

2. **Column Cleaning**
   - Dropped `id` and standardized column names to lowercase snake_case.

3. **Feature Selection**
   - Chose features with known clinical relevance: `age`, `hypertension`, `heart_disease`, `avg_glucose_level`, `bmi`, `smoking_status`, and `ever_married`.

4. **Feature Engineering – Binning**
   - Binned continuous variables using medical thresholds:
     - Age: `<25, 25–44, 45–64, 65–79, 80+`.
     - BMI: `Underweight, Normal, Overweight, Obese`.
     - Glucose: `<70, 70–84, 85–99, 100–109, 110–125, 126–139, ≥140`.

5. **Encoding & Standardization**
   - One-hot encoded binned and categorical features.
   - Standardized original continuous features to preserve numeric precision for linear models.

6. **Standardlization**
   - Although `age`, `bmi`, and `avg_glucose_level` were binned into categorical groups for medical interpretability, the original continuous variables were retained in the dataset. This was done to preserve numerical precision and avoid excessive dimensionality from one-hot encoding. 

7. **Imbalanced Class Handeling：Data-Level Methods**

   Based on He & Garcia (2009), instead of adjusting each classifier's internal weighting (`class_weight`) or modifying the loss function,imbalance can be addressed at three levels：data-level，algorithm-level，evaluation-level，He & Garcia recommend data-level approaches as the most effective and generalizable.
   - Use **ADASYN (Adaptive Synthetic Sampling)**, which is preferable to SMOTE because it focuses more on **difficult-to-classify** instances.
   - **Compare three resampling strategies**: SMOTE, ADASYN, and SMOTEENN, and record the performance differences.
     
---

## **Model Selection Strategy**
*(Based on Hutter et al., 2019; Dong et al., 2020)*


### Candidate Models

#### Single Base Models
- **Logistic Regression** – Strong interpretability; serves as a baseline.
- **Random Forest** – High robustness; well-suited for small to medium-sized datasets.
- **XGBoost** – High accuracy; particularly effective for imbalanced classification tasks.

#### Ensemble Strategies (Dong et al., 2020)
- **VotingClassifier (soft voting)** – Combines predictions from multiple models.
- **StackingClassifier** – Integrates multiple base learners with a meta-learner (e.g., Logistic Regression).

### AutoML Framework
- Employed `ImbPipeline`, `StratifiedKFold`, and `GridSearchCV`.
- Nested cross-validation for unbiased hyperparameter tuning.

### Evaluation Metrics
- **Primary**: Recall, F1-score, AUC.
  - **Recall** – For detecting stroke cases (minority class).
  - **AUC** – For evaluating overall model stability and discrimination ability.
- **Secondary**: Precision, G-Mean.
- **Visualization**: Confusion matrices, ROC/PR curves.



---

## **Experimental Design**  
*(Based on He & Garcia, 2009; Dong et al., 2020; Hutter et al., 2019)*

### STEP1: Baseline Model – Logistic Regression
- **Aim:** Establish a minimal benchmark without resampling.
- **Result:** Accuracy = 0.957, Recall = 0.0048, F1 = 0.0093. Model could not detect strokes.

### STEP 2: Resampling Strategy Comparison
- **Aim:** Assess effectiveness of data-level methods using Random Forest.
  - Original Data (no resampling)
  - **SMOTE**
  - **ADASYN**
  - **SMOTEENN**
- **Result:** SMOTEENN achieved highest F1 (0.214) and G-Mean (0.667); ADASYN achieved highest Recall (0.593).

### STEP 3:Classifier Comparison under Best Resampling
- **Aim:** Compare Random Forest, XGBoost, and Stacking under ADASYN and SMOTEENN.
  - Random Forest
  - XGBoost
  - VotingClassifier
  - StackingClassifier
- Use consistent **5-fold Stratified Cross-Validation** for performance comparison.
- - **Result:** SMOTEENN + Random Forest was most balanced (F1 = 0.214, Recall = 0.526); stacking under ADASYN underperformed.

### STEP 4:  Threshold Tuning

- **Aim:** Adjust the classification threshold to improve recall-F1 tradeoff on the best model.

- **Method:** Use precision-recall curve to select the optimal threshold.

- **Result:** F1 improved from 0.214 to 0.464 with threshold = 0.743. Recall increased to 0.63, with acceptable precision.


---

## **Conclusion and Future Improvements**

The proposed pipeline successfully improved stroke detection on highly imbalanced data, elevating F1-score from 0.0093 (baseline) to 0.464 and recall increased to 0.63  (best model). SMOTEENN combined with Random Forest offered the best trade-off between precision and recall. To further enhance performance and clinical applicability:

- **Threshold Optimization:** Fine-tune the classification threshold via Precision-Recall analysis to maximize clinically relevant metrics.
- **Feature Refinement:** Use SHAP-driven feature selection to eliminate weak predictors and engineer interaction terms.
- **Advanced Models:** Experiment with LightGBM or CatBoost, which may better capture sparse patterns.
- **Clinical Validation:** Collaborate with healthcare professionals to validate model predictions and integrate domain knowledge.


---

**References**  
- He, H. & Garcia, E. A. (2009). Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering*, 21(9), 1263–1284.  
- Hutter, F., Kotthoff, L. & Vanschoren, J. (2019). *Automated Machine Learning: Methods, Systems, Challenges*. Springer Nature.  
- Dong, X., Yu, Z., Cao, W., Shi, Y. & Ma, Q. (2020). A survey on ensemble learning. *Frontiers of Computer Science*, 14, 241–258.



## **Exploration Data Aanalysis**

### **Shape, Missing Data, Data Type**

In [ ]:
import pandas as pd  
import numpy as np  

# Read the stroke dataset into a DataFrame
df = pd.read_csv('COURSEWORK2/stroke-data.csv')

print(f"Dataset Shape:{df.shape}")


# Missing NaN values
missing_df = df.isnull().sum().to_frame(name='Missing Count')
missing_df['Missing %'] = (missing_df['Missing Count'] / len(df)) * 100

# Count 'Unknown' in smoking_status
unknown_count = df['smoking_status'].value_counts().get('Unknown', 0)
unknown_percent = (unknown_count / len(df)) * 100

# Create DataFrame for 'Unknown' as missing
unknown_row = pd.DataFrame({
    'Missing Count': [unknown_count],
    'Missing %': [unknown_percent]
}, index=['smoking_status_Unknown'])

# Combine NaN-based missing and 'Unknown'-based missing
combined_missing_df = pd.concat([missing_df[missing_df['Missing Count'] > 0], unknown_row])

# Print result
print('==MISSING DATA==')
print(combined_missing_df)

      # type
print('==DATA TYPE==')
print(df.dtypes)


### **Descriptive Statistics of Target Variable and Numerical Features**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print(df.describe())

# delete ID column
if 'id' in df.columns:
    df = df.drop(columns='id')

stroke_counts = df['stroke'].value_counts()
hyper_counts = df['hypertension'].value_counts()
heart_counts = df['heart_disease'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Stroke pie chart
axes[0].pie(
    stroke_counts,
    autopct='%1.1f%%',
    labels=['No Stroke', 'Stroke'],
    colors=['skyblue', 'salmon'],
    startangle=90,
    shadow=True
)
axes[0].set_title('Stroke Proportion')

# Hypertension pie chart
axes[1].pie(
    hyper_counts,
    autopct='%1.1f%%',
    labels=['No Hypertension', 'Hypertension'],
    colors=['skyblue', 'salmon'],
    startangle=90,
    shadow=True
)
axes[1].set_title('Hypertension Proportion')

# Heart Disease pie chart
axes[2].pie(
    heart_counts,
    autopct='%1.1f%%',
    labels=['No Heart Disease', 'Heart Disease'],
    colors=['skyblue', 'salmon'],
    startangle=90,
    shadow=True
)
axes[2].set_title('Heart Disease Proportion')

for ax in axes:
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

# New list for visulization without target variavle 'stroke'
numerical_cols =['age', 'avg_glucose_level', 'bmi']
categorical_cols = df.select_dtypes(include=['object']).columns


numerical_cols = [col for col in numerical_cols if col in df.columns]

# Reshape numerical dataset to long format for FacetGrid
df_melted_num = df.melt(id_vars='stroke', value_vars=numerical_cols)

# Define histogram plot function for numerical features
def histplot_with_kde(data, x, **kwargs):
    ax = plt.gca()
    sns.histplot(data=data, x=x, kde=True, bins=30, ax=ax)
    ax.set_title(f'Distribution of {data["variable"].iloc[0]}')
    ax.set_xlabel(x)
    ax.set_ylabel('Frequency')

# Create FacetGrid for numerical distributions
d = sns.FacetGrid(df_melted_num, col='variable', col_wrap=3, sharex=False, sharey=False, height=3)
d.map_dataframe(histplot_with_kde, x='value')

plt.tight_layout()
plt.show()

### **Frequency Analysis of Categorical Features**

In [ ]:
# Reshape the categorical dataset to long format for FacetGrid
import seaborn as sns
df_melted_cat = df.melt(id_vars='stroke', value_vars=categorical_cols)

def countplot_with_labels(data, x, **kwargs):
    ax = kwargs.get("ax", plt.gca()) 
    sns.countplot(data=data, x=x, ax=ax, **kwargs)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30)
    for container in ax.containers:
        ax.bar_label(container, fontsize=8)

# FacetGrid for categorical
g = sns.FacetGrid(df_melted_cat, col='variable', col_wrap=3, sharex=False, sharey=False, height=3)
g.map_dataframe(countplot_with_labels, x='value')

g.set_titles('{col_name}')
plt.tight_layout()
plt.show()

### **Correlation Analysis of Target Variable and Categorical Variables**

In [ ]:
grouped_means = df.groupby('stroke')[numerical_cols].mean().T
grouped_means.columns = ['No Stroke (0)', 'Stroke (1)']

# 
grouped_means.plot(kind='bar', figsize=(6, 3))
plt.title('Mean Comparison of Numerical Features by Stroke Outcome')
plt.ylabel('Mean Value')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

for col in categorical_cols:
    stroke_rate = pd.crosstab(df[col], df['stroke'], normalize='index') * 100
    stroke_rate.plot(kind='bar', stacked=True, figsize=(5, 3), colormap='coolwarm')
    plt.title(f'Stroke Distribution by {col}')
    plt.ylabel('Percentage')
    plt.xlabel(col)
    plt.xticks(rotation=30)
    plt.legend(title='Stroke', labels=['No', 'Yes'])
    plt.tight_layout()
    plt.show()

### **Correlation Analysis of Target Variable and Numerical Data**

In [ ]:
# The relationship between AGE vs STROKE
sns.regplot(x='age', y='stroke', data=df, logistic=True, ci=None)
plt.title('Logistic Curve: Age vs Stroke Probability')
plt.xlabel('Age')
plt.ylabel('Stroke (0/1)')
plt.show()

# The relationship between GLUCOSE vs STROKE
sns.regplot(x='avg_glucose_level', y='stroke', data=df, logistic=True, ci=None)
plt.title('Logistic Curve: Glucose Level vs Stroke Probability')
plt.xlabel('Avg Glucose Level')
plt.ylabel('Stroke (0/1)')
plt.show()

# The relationship between BMI vs STROKE
sns.regplot(x='bmi', y='stroke', data=df[df['bmi'].notna()], logistic=True, ci=None)
plt.title('Logistic Curve: BMI vs Stroke Probability')
plt.xlabel('BMI')
plt.ylabel('Stroke (0/1)')
plt.show()

### **Preliminary Outlier Detection**

In [ ]:
# Reshape the dataset to long format for FacetGrid
df_melted = df.melt(id_vars='stroke', value_vars=[col for col in numerical_cols if col != 'stroke'])

# Create a FacetGrid for boxplots
g = sns.FacetGrid(df_melted, col='variable', col_wrap=3, sharex=False, sharey=False, height=3)
g.map(sns.boxplot, 'value')
g.set_titles('{col_name}')
plt.tight_layout()
plt.show()


## **Pre-processing Data**

In [ ]:
import re 
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

# Row Filtering
df = df[df['gender'] != 'Other']
df= df.dropna()

# Columns Cleaning
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)
    
df.columns = df.columns.str.lower().map(lambda s: re.sub(r'[^0-9a-z_]', '_', s))

X = df.drop(columns='stroke')
y = df['stroke']


# Feature Engineering- binning
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing  import (
    StandardScaler, OneHotEncoder, FunctionTransformer
)


# Categorical Feature Encoding

# Feature Engineering – binning function
def make_bins(X):
    df = pd.DataFrame(X, columns=['age', 'bmi', 'avg_glucose_level'])
    df['age_bin'] = pd.cut(df['age'], [0, 24, 44, 64, 79, 150], labels=['<25', '25-44', '45-64', '65-79', '80+'])
    df['bmi_bin'] = pd.cut(df['bmi'], [0, 18.5, 24.9, 29.9, 100], labels=['Underweight', 'Normal', 'Overweight', 'Obese'])
    df['glu_bin'] = pd.cut(df['avg_glucose_level'], [0, 70, 84, 99, 109, 125, 139, float('inf')],
                           labels=['<70', '70–84', '85–99', '100–109', '110–125', '126–139', '≥140'])
    return df[['age_bin', 'bmi_bin', 'glu_bin']]

bin_encoder= ImbPipeline([
    ('bin',FunctionTransformer(make_bins,validate=False)),
    ('ohe',OneHotEncoder(handle_unknown='ignore'))
])

numeric_features = ['age', 'bmi', 'avg_glucose_level']

preprocessor = ColumnTransformer([
    
    ('bins', bin_encoder, ['age', 'bmi', 'avg_glucose_level']),
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), [
        'gender', 'ever_married', 'work_type',
        'residence_type', 'smoking_status',
        'hypertension', 'heart_disease'
    ])
])


df.head()


## **Model Selection and Expiriental Design**

#### **STEP 1: Baseline Model:-Logistic Regression** (Cleaned Data, No Sampling)
- **Aim:** To establish a baseline using Logistic Regression on the imbalanced dataset
- **Result:**  While accuracy appears high, the model fails to detect stroke cases, as indicated by extremely low recall and F1-score.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV,cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
import re


# Baseline model pipeline
baseline_pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(max_iter=1000))
])

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score


# Define inner and outer CV
inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Set hyperparameter grid 
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

# Nested CV
clf = GridSearchCV(estimator=baseline_pipeline, param_grid=param_grid, cv=inner_cv, scoring='f1', n_jobs=-1)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
nested_score = cross_validate(clf, X, y, cv=outer_cv, scoring= scoring,return_estimator=True)

print("=== Baseline Logistic Regression (No Sampling) ===")
for metric in scoring:
    print(f"{metric}: {nested_score[f'test_{metric}'].mean():.4f}")

# Confusion Matrix (based on fold 0)
print("\n=== Confusion Matrix (Fold 0) ===")
model_fold0 = nested_score['estimator'][0]
y_pred_fold0 = model_fold0.predict(X)
cm = confusion_matrix(y, y_pred_fold0)
ConfusionMatrixDisplay(cm, display_labels=['No Stroke', 'Stroke']).plot()



#### **STEP 2: Resampling Strategy Comparison – Fixed Model: Random Forest**
- **Aim:** To compare the effects of different resampling methods (Original, SMOTE, ADASYN, SMOTEENN) on model performance using a fixed Random Forest classifier with nested cross-validation.
- **Results:** Compared to the original data, all resampling techniques significantly improved stroke detection, with SMOTEENN achieving the best overall F1-score and ADASYN yielding the highest recall and geometric mean.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer
from imblearn.metrics import geometric_mean_score
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTEENN


# Define resampling strategies
samplers = {
    'Original': None,
    'SMOTE': SMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

# Define fixed model
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)


# Define outer and inner CV
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)

# Define scoring
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'gmean': make_scorer(geometric_mean_score)
}

# Define Random Forest hyperparameter grid
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [5, 10],
    'clf__min_samples_split': [2, 5],
    'clf__min_samples_leaf': [1, 3]
}

# Compare each sampling strategy
results = []

for name, sampler in samplers.items():
    if sampler:
        pipeline = ImbPipeline([
            ('pre', preprocessor),
            ('sampler', sampler),
            ('clf', RandomForestClassifier(random_state=42))
        ])
    else:
        pipeline = Pipeline([
            ('pre', preprocessor),
            ('clf', RandomForestClassifier(random_state=42))
        ])
    
    # Nested CV with grid search in the inner loop
    grid = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=inner_cv, scoring='f1', n_jobs=-1)

    cv_result = cross_validate(grid, X, y, cv=outer_cv, scoring=scoring, return_estimator=True)
    result = {metric: cv_result[f'test_{metric}'].mean() for metric in scoring}
    result['strategy'] = name
    result['best_params'] = [est.best_params_ for est in cv_result['estimator']]
    results.append(result)

# Save results
results_df = pd.DataFrame(results)
print(results_df)


#### **STEP 3: Model Comparison**
- **Aim:** To compare the performance of different classifiers **(Random Forest, XGBoost, Stacking)** under the two best resampling strategies identified in Step 2 — **ADASYN and SMOTEENN** — using nested cross-validation with automated hyperparameter tuning.

- **Result:** 
  - **SMOTEENN + Random Forest** achieved the highest F1-score (0.214), G-Mean (0.667), and a strong recall (0.526), making it the most balanced and effective combination.
  - **ADASYN + Random Forest** yielded the highest recall (0.593), it suffered from low precision (0.115), indicating a higher false positive rate.
  - **Stacking** performed inconsistently, particularly under ADASYN, where both F1-score and recall dropped sharply.


In [ ]:
from sklearn.ensemble import  StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier



# CV + scoring
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'gmean': make_scorer(geometric_mean_score)
}

# Define resampling methods
samplers = {
    'ADASYN': ADASYN(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

# Define model configs
model_configs = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'clf__n_estimators': [100, 200],
            'clf__max_depth': [5, 10],
            'clf__min_samples_split': [2, 5],
            'clf__min_samples_leaf': [1, 3]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(use_label_encoder=False, eval_metric='auc', random_state=42),
        'params': {
            'clf__n_estimators': [100, 200],
            'clf__max_depth': [3, 5],
            'clf__learning_rate': [0.01, 0.1],
            'clf__scale_pos_weight': [1, 5, 10]
        }
    },
    'Stacking': {
        'model': StackingClassifier(
            estimators=[
                ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
                ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='auc', random_state=42))
            ],
            final_estimator=LogisticRegression(),
            cv=3,
            n_jobs=-1
        ),
        'params': {
            'clf__final_estimator__C': [0.01, 0.1, 1, 10],
            'clf__final_estimator__penalty': ['l2'],
            'clf__final_estimator__solver': ['lbfgs']
        }
    }
}

# Run nested CV over all sampler-model combinations
results = []

for sampler_name, sampler_obj in samplers.items():
    for model_name, config in model_configs.items():
        print(f">>> Evaluating: {sampler_name} + {model_name}")
        
        pipeline = ImbPipeline([
            ('pre', preprocessor),
            ('sampler', sampler_obj),
            ('clf', config['model'])
        ])
        
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=config['params'],
            cv=inner_cv,
            scoring='f1',
            n_jobs=-1
        )
        
        cv_result = cross_validate(
            grid, X, y, cv=outer_cv, scoring=scoring, return_estimator=True
        )
        
        result = {metric: cv_result[f'test_{metric}'].mean() for metric in scoring}
        result['sampler'] = sampler_name
        result['model'] = model_name
        result['best_params'] = [est.best_params_ for est in cv_result['estimator']]
        results.append(result)

# Format results
results_df = pd.DataFrame(results)
ordered_cols = ['sampler', 'model', 'f1', 'recall', 'gmean', 'roc_auc', 'accuracy', 'precision', 'best_params']
results_df = results_df[ordered_cols]

# Show final comparison
print("\n=== STEP 3: Model Comparison under ADASYN and SMOTEENN ===")
print(results_df.sort_values(by='f1', ascending=False))


#### **STEP 4: Targeted Improvement – Threshold Tuning for Best Model**
- **Aim:** To enhance the performance of the best model identified in Step 3 (SMOTEENN + RandomForest) by tuning the decision threshold to improve the balance between precision and recall.
- **Result:** **SMOTEENN + Random Forest** optimal threshold = 0.743; F1-score increased to 0.464, with recall = 0.63 and precision = 0.37. The confusion matrix confirmed improved stroke detection.



In [ ]:

from sklearn.metrics import precision_recall_curve, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Step 1: Use the best model from Step 3 – SMOTEENN + RandomForest

raw_params = results_df.query("sampler == 'SMOTEENN' and model == 'RandomForest'")['best_params'].values[0][0]
best_rf_model = {k.replace('clf__', ''): v for k, v in raw_params.items()}

# Final model
final_pipeline = ImbPipeline([
    ('pre', preprocessor),
    ('sampler', SMOTEENN(random_state=42)),
    ('clf', RandomForestClassifier(**best_rf_model, random_state=42))
])

final_pipeline.fit(X, y)

# Step 2: Predict probabilities
y_proba = final_pipeline.predict_proba(X)[:, 1]

# Step 3: Compute precision-recall curve and F1
precision, recall, thresholds = precision_recall_curve(y, y_proba)
f1s = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1s)
best_threshold = thresholds[best_idx]
best_f1 = f1s[best_idx]

print(f"Best threshold: {best_threshold:.3f} — F1-score: {best_f1:.4f}")

# Step 4: Plot F1 vs threshold
plt.figure(figsize=(8, 5))
plt.plot(thresholds, f1s[:-1], label='F1-score')
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best Threshold = {best_threshold:.2f}')
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score vs Classification Threshold")
plt.legend()
plt.grid()
plt.show()

# Step 5: Predict with tuned threshold
y_pred_custom = (y_proba >= best_threshold).astype(int)

# Step 6: Show confusion matrix
cm = confusion_matrix(y, y_pred_custom)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Stroke", "Stroke"]).plot()
